# Clinical Trial Footprint vs. Population Structure

**Course:** Data Science — Regular Exam, March 2026  
**Author:** Momchil Ivanov  
**Deadline:** 28 April 2026, 16:00h  
**Repository:** [github.com/Momchil-Ivanov/Data-Science-Mar-2026](https://github.com/Momchil-Ivanov/Data-Science-Mar-2026/tree/clinical-trial-footprint-vs-population-structure)

---

## Section 0 — Introduction & Problem Formulation

### 0.1 Motivation

Clinical trials are the gold standard for establishing the safety and efficacy of new medical treatments. Where these trials are conducted matters enormously: countries that host more trials gain earlier access to experimental therapies, attract research investment, and build local scientific capacity. Countries that are systematically underrepresented risk falling behind in medical innovation and having treatments developed without sufficient data from their populations.

This project asks a simple but revealing question: **is the global distribution of clinical trials proportional to where people actually live?** Or is it skewed — concentrated in wealthy, high-income countries regardless of population size?

We use two independent public data sources:

| Source | What it provides | Access |
|---|---|---|
| **ClinicalTrials.gov** | Trial recruitment site counts per country, phase, status, start year | REST API v2 — `https://clinicaltrials.gov/api/v2/` |
| **World Bank Indicators** | Population, GDP per capita, world region, income group | REST API v2 — `https://api.worldbank.org/v2/` |

Both sources are freely accessible without authentication and are maintained by authoritative international institutions (U.S. National Library of Medicine and the World Bank Group, respectively).

> **Reproducibility note:** No data files are committed to this repository. All data is fetched from the APIs above at runtime. Running Section 1 from top to bottom will recreate the full dataset.

---

### 0.2 Research Questions

We investigate four concrete questions. The appropriate statistical methods and formal hypotheses for each will be introduced in the relevant analysis section, after the data has been inspected.

1. **Q1 — Income groups:** How many trials per million inhabitants do different countries have, and how does this differ between income groups?
2. **Q2 — Population scaling:** Does a larger population proportionally lead to more trials, or is the relationship sub- or super-linear?
3. **Q3 — Regional inequality:** Is there a systematic difference in trial density across world regions, and how concentrated is the global distribution?
4. **Q4 — Temporal trends:** How has the picture changed over time (2000–2024), and is the gap between income groups growing or shrinking?

---

### 0.3 Prior Work

The unequal global distribution of clinical trials has been documented under the label of the **"10/90 gap"** — roughly 90 % of global health research funding addresses diseases affecting only 10 % of the world's population (Global Forum for Health Research, 2000). Viergever & Li (2015) mapped trial density globally using the WHO ICTRP registry and found a strong positive correlation with GDP per capita. Drain et al. (2014) showed that high-income countries account for the large majority of trial sites despite representing a minority of global disease burden.

This project extends that work by explicitly testing whether the population-to-trial relationship is proportional, and by applying distributional inequality tools (Lorenz curve, Gini coefficient) to quantify concentration rather than relying on regional averages alone.

In [4]:
import requests
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pycountry
import statsmodels.formula.api as smf
from scipy import stats
from scipy.stats import kruskal, mannwhitneyu
from itertools import combinations
import plotly.express as px

print('All imports OK')

ModuleNotFoundError: No module named 'pycountry'

## Section 1 — Data Ingestion

### 1.1 ClinicalTrials.gov API v2

We fetch trial metadata directly from the ClinicalTrials.gov REST API v2. No files are stored in the repository — data is recreated at runtime on every run.

**What we extract per study:**
- `nctId` — unique trial identifier
- `startDate` — trial start date (we keep the year)
- `locations[].country` — every country where the trial recruits participants

**Unit of measurement:** a single multinational trial with sites in 30 countries contributes **1 count to each of those 30 countries**. We therefore measure *trial-site presence per country*, not globally unique trials. This is a deliberate choice: it reflects the research activity actually experienced by a given country's population.

**Filters applied at fetch time:**
- `overallStatus`: `COMPLETED`, `ACTIVE_NOT_RECRUITING`, `RECRUITING` — excludes withdrawn/terminated studies with no recruitment activity.

In [ ]:
CT_BASE = "https://clinicaltrials.gov/api/v2/studies"

CT_PARAMS = {
    "format": "json",
    "pageSize": 1000,
    "fields": "NCTId,StartDate,LocationCountry",
    "filter.overallStatus": "COMPLETED,ACTIVE_NOT_RECRUITING,RECRUITING",
}

def fetch_clinicaltrials() -> pd.DataFrame:
    """Page through ClinicalTrials.gov API v2 and return one row per
    (nct_id, country, start_year) triplet."""
    records = []
    params = CT_PARAMS.copy()

    while True:
        resp = requests.get(CT_BASE, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()

        for study in data.get("studies", []):
            proto = study.get("protocolSection", {})
            nct_id = proto.get("identificationModule", {}).get("nctId", "")
            start_raw = proto.get("statusModule", {}).get("startDateStruct", {}).get("date", "")
            start_year = int(start_raw[:4]) if start_raw and len(start_raw) >= 4 else None
            locations = proto.get("contactsLocationsModule", {}).get("locations", [])
            countries = {loc.get("country") for loc in locations if loc.get("country")}
            for country in countries:
                records.append({"nct_id": nct_id, "country": country, "start_year": start_year})

        next_token = data.get("nextPageToken")
        if not next_token:
            break
        params["pageToken"] = next_token
        time.sleep(0.2)  # stay within API rate limits

    df = pd.DataFrame(records)
    print(f"Fetched {df['nct_id'].nunique():,} unique studies → {len(df):,} (study, country) pairs")
    print(f"Unique country names in raw data: {df['country'].nunique()}")
    return df

ct_raw = fetch_clinicaltrials()
ct_raw.head()